# 🔬 Módulo 5: El Detective de Código — Sampling Profiler
## Notebook de Conocimiento Guiado

**Objetivo:** Entender cómo funcionan los profilers de muestreo, cómo reconstruyen
la línea de tiempo de ejecución y cómo generan flame graphs.

**Lore:** Eres el Detective de Código de la Flota Galáctica. Tu arma: el Profiler de
Muestreo Cuántico, capaz de ver exactamente dónde se pierde el tiempo.

| Sección | Concepto |
|---------|---------|
| 📚 Teoría | Sampling vs instrumentación, call stacks |
| 🔨 Demo | divergencia de stacks, reconstrucción de timeline |
| ✏️ Ejercicio 1 | `HotFunctions` — top-N funciones |
| ✏️ Ejercicio 2 | `CallGraph` — quién llama a quién |
| 🎯 Quiz | perf, py-spy, eBPF, dtrace |

## 📚 Parte 1: Instrumentación vs Muestreo

| Técnica | Cómo funciona | Ventaja | Desventaja |
|---------|--------------|---------|------------|
| **Instrumentación** | Inserta código al inicio/fin de cada función | 100% preciso | Overhead alto, altera el comportamiento |
| **Muestreo (Sampling)** | Interrumpe el programa N veces/segundo y captura el call stack | Bajo overhead (<5%) | No captura llamadas muy cortas |

**El truco del muestreo:**
Si una función aparece en 80 de 100 muestras, está ocupando ~80% del tiempo de CPU.
No necesitamos medir con exactitud: la estadística nos da la respuesta.

### Herramientas reales:
- **Linux**: `perf record -g`, `eBPF/bpftrace`
- **Python**: `py-spy`, `pyinstrument`, `cProfile`
- **Java**: `async-profiler`, `JFR`
- **Go**: `pprof`
- **Mac**: Instruments / XCode Profiler
- **Chrome**: `chrome://tracing`

In [ ]:
# 🔍 Modelando el Call Stack
from dataclasses import dataclass, field
from typing import List

@dataclass
class StackFrame:
    """Un frame del call stack — una función en ejecución."""
    funcion: str
    linea: int = 0

@dataclass
class StackSample:
    """Una 'foto' del call stack tomada en un momento dado."""
    frames: List[StackFrame]   # [frame_raíz, ..., frame_actual]
    timestamp: float = 0.0
    
    @property
    def depth(self): return len(self.frames)
    
    def __repr__(self):
        return " → ".join(f.funcion for f in self.frames)

# Simulamos las muestras de un profiler
muestras = [
    StackSample([StackFrame("main"), StackFrame("procesar"), StackFrame("leer_datos")], 0.0),
    StackSample([StackFrame("main"), StackFrame("procesar"), StackFrame("leer_datos")], 0.1),
    StackSample([StackFrame("main"), StackFrame("procesar"), StackFrame("calcular")], 0.2),
    StackSample([StackFrame("main"), StackFrame("procesar"), StackFrame("calcular")], 0.3),
    StackSample([StackFrame("main"), StackFrame("procesar"), StackFrame("guardar")], 0.4),
    StackSample([StackFrame("main"), StackFrame("reportar")], 0.5),
]

print("Muestras capturadas:")
for i, m in enumerate(muestras):
    print(f"  t={m.timestamp:.1f}s: {m}")

## 🏗️ Parte 2: Algoritmo de Divergencia

Para reconstruir la timeline de ejecución, comparamos stacks consecutivos:

```
Muestra anterior:  main → procesar → leer_datos
Muestra actual:    main → procesar → calcular

Punto de divergencia: índice 2 (leer_datos ≠ calcular)

Eventos generados:
  - END leer_datos (se terminó)
  - START calcular (comenzó)
```

In [ ]:
# 🔨 IMPLEMENTACIÓN: Algoritmo de Divergencia

def find_divergence_point(prev: StackSample, curr: StackSample) -> int:
    """
    Encuentra el índice donde dos stacks consecutivos divergen.
    Compara frame a frame desde la raíz (índice 0).
    Retorna el primer índice donde los frames difieren.
    """
    min_depth = min(prev.depth, curr.depth)
    for i in range(min_depth):
        if prev.frames[i].funcion != curr.frames[i].funcion:
            return i
    return min_depth   # Uno es prefijo del otro

# Test
s1 = StackSample([StackFrame("main"), StackFrame("procesar"), StackFrame("leer_datos")])
s2 = StackSample([StackFrame("main"), StackFrame("procesar"), StackFrame("calcular")])
s3 = StackSample([StackFrame("main"), StackFrame("reportar")])

print(f"Divergencia s1 vs s2: índice {find_divergence_point(s1, s2)}")  # 2
print(f"Divergencia s2 vs s3: índice {find_divergence_point(s2, s3)}")  # 1


def reconstruct_timeline(muestras: List[StackSample]) -> List[dict]:
    """
    Reconstruye la timeline de eventos START/END comparando muestras consecutivas.
    """
    eventos = []
    
    if not muestras:
        return eventos
    
    # Primera muestra: todas las funciones inician
    for frame in muestras[0].frames:
        eventos.append({"tipo": "START", "funcion": frame.funcion,
                        "timestamp": muestras[0].timestamp})
    
    for i in range(1, len(muestras)):
        prev, curr = muestras[i-1], muestras[i]
        div = find_divergence_point(prev, curr)
        
        # Funciones que terminaron (del punto de divergencia hacia abajo)
        for j in range(prev.depth - 1, div - 1, -1):
            eventos.append({"tipo": "END", "funcion": prev.frames[j].funcion,
                            "timestamp": curr.timestamp})
        
        # Funciones que iniciaron (del punto de divergencia hacia arriba)
        for j in range(div, curr.depth):
            eventos.append({"tipo": "START", "funcion": curr.frames[j].funcion,
                            "timestamp": curr.timestamp})
    
    # Última muestra: todas las funciones terminan
    for frame in reversed(muestras[-1].frames):
        eventos.append({"tipo": "END", "funcion": frame.funcion,
                        "timestamp": muestras[-1].timestamp + 0.1})
    
    return eventos

timeline = reconstruct_timeline(muestras)
print("\nTimeline reconstruida:")
for ev in timeline[:10]:
    print(f"  t={ev['timestamp']:.1f}s  {ev['tipo']:5s}  {ev['funcion']}")

In [ ]:
# 🔨 IMPLEMENTACIÓN: Flame Graph (agregación)

from collections import defaultdict

def build_flame_graph(muestras: List[StackSample]) -> dict:
    """
    Construye un flame graph contando cuántas muestras contiene cada función.
    Un flame graph muestra: las funciones más 'anchas' son las que más tiempo consumen.
    
    Returns: dict[str, int] — {funcion: número_de_muestras}
    """
    conteos = defaultdict(int)
    for muestra in muestras:
        for frame in muestra.frames:
            conteos[frame.funcion] += 1
    return dict(conteos)

def print_flame_graph(conteos: dict, total_muestras: int):
    """Imprime el flame graph en ASCII."""
    print("\n🔥 Flame Graph (% tiempo de CPU):")
    print("=" * 50)
    for funcion, n in sorted(conteos.items(), key=lambda x: -x[1]):
        pct = n / total_muestras * 100
        barra = "█" * int(pct / 2)
        print(f"  {funcion:<20} {barra:<25} {pct:.1f}% ({n}/{total_muestras})")

conteos = build_flame_graph(muestras)
print_flame_graph(conteos, len(muestras))

## ✏️ Ejercicio 1: `HotFunctions`

**Problema:** Implementa una función que retorne las N funciones más "calientes"
(las que aparecen en más muestras como el frame más profundo — la hoja del call stack).

In [ ]:
# ✏️ EJERCICIO 1 — Completa el TODO
import heapq

def hot_functions(muestras: List[StackSample], top_n: int = 3) -> List[tuple]:
    """
    Retorna las top_n funciones hoja (más profundas) más frecuentes.
    
    Returns:
        list[(funcion, count)] ordenada por frecuencia descendente
    """
    # TODO: Cuenta solo el frame MÁS PROFUNDO de cada muestra (el que más trabaja)
    pass

resultado = hot_functions(muestras, top_n=3)
print("Top 3 funciones hoja:")
for funcion, count in resultado:
    print(f"  {funcion}: {count} muestras")

In [ ]:
# 💡 SOLUCIÓN — Ejercicio 1

def hot_functions(muestras: List[StackSample], top_n: int = 3) -> List[tuple]:
    conteos = defaultdict(int)
    for muestra in muestras:
        if muestra.frames:
            # Solo el frame MÁS PROFUNDO (hoja del call stack)
            conteos[muestra.frames[-1].funcion] += 1
    
    return heapq.nlargest(top_n, conteos.items(), key=lambda x: x[1])

resultado = hot_functions(muestras, top_n=3)
print("Top 3 funciones hoja (más tiempo en CPU):")
for funcion, count in resultado:
    print(f"  {funcion}: {count} muestras ({count/len(muestras)*100:.0f}%)")

## ✏️ Ejercicio 2: `CallGraph`

**Problema:** Construye un grafo de llamadas (quién llama a quién) desde las muestras.

In [ ]:
# ✏️ EJERCICIO 2 — Completa el TODO

def build_call_graph(muestras: List[StackSample]) -> dict:
    """
    Construye un grafo de llamadas.
    
    Returns:
        dict[str, dict[str, int]] — {caller: {callee: count}}
    """
    # TODO: Para cada muestra, registra cada par (frames[i], frames[i+1]) como llamada
    pass

grafo = build_call_graph(muestras)
print("Call graph:")
for caller, callees in sorted(grafo.items()):
    for callee, count in sorted(callees.items()):
        print(f"  {caller} → {callee}: {count}x")

In [ ]:
# 💡 SOLUCIÓN — Ejercicio 2

def build_call_graph(muestras: List[StackSample]) -> dict:
    grafo = defaultdict(lambda: defaultdict(int))
    for muestra in muestras:
        for i in range(len(muestra.frames) - 1):
            caller = muestra.frames[i].funcion
            callee = muestra.frames[i + 1].funcion
            grafo[caller][callee] += 1
    return {k: dict(v) for k, v in grafo.items()}

grafo = build_call_graph(muestras)
print("Call graph (caller → callee: frecuencia):")
for caller, callees in sorted(grafo.items()):
    for callee, count in sorted(callees.items(), key=lambda x: -x[1]):
        print(f"  {caller} → {callee}: {count}x")

## 🎯 Quiz — Preguntas de Entrevista

**P1:** ¿Por qué un sampling profiler tiene menor overhead que uno de instrumentación?
> **R:** Solo interrumpe el programa N veces por segundo (ej: 100Hz = 1% overhead),
> en lugar de añadir código a cada función (que puede ser 10-100x más costoso).

**P2:** ¿Qué es un flame graph y cómo se lee?
> **R:** Eje X = tiempo/muestras (ancho = % CPU), eje Y = profundidad del call stack.
> Las funciones más anchas en la base son los cuellos de botella principales.
> No confundir con "más caliente" = más coloreado (solo es estético).

**P3:** ¿Cuándo usarías `perf` vs `py-spy`?
> **R:** `perf`: para código C/C++, queries del kernel, profiling de sistema completo.
> `py-spy`: para Python sin modificar el proceso (no requiere reiniciar el servicio).

**P4:** ¿Qué es eBPF y por qué revolucionó el profiling en Linux?
> **R:** eBPF permite ejecutar programas sandboxed en el kernel sin módulos de kernel.
> Permite profiling, tracing y networking observability con overhead mínimo y seguridad garantizada.